In [ ]:
from langchain_ollama import ChatOllama
llm = ChatOllama(
    model="llama3.1:8b"   # or qwen3:8b, mistral, etc.
)

In [ ]:
def multiply(a: int,b:int) -> int:
    """Multiply a and b.
    args:
        a: first number
        b: second number"""
    return a*b

def add(a:int,b:int) -> int:
    """ Add a and b.
    args:
        a: first number
        b: second number"""
    return a+b
def divide(a:int,b:int) -> int:
    """ Divide a and b.
    args:
        a: first number
        b: second number"""
    return a/b

tools =[add, multiply, divide]
llm = ChatOllama(
    model="llama3.1:8b"   # or qwen3:8b, mistral, etc.
)
llm_with_tools =llm.bind_tools(tools)


In [ ]:
from langgraph.graph import MessagesState
from langchain_core.messages import AIMessage, HumanMessage,SystemMessage
sys_msg = SystemMessage(content = "you are a helpful assistant tasked with performing arithmetic on a set of inputs.")

def assistant(state:MessagesState):
    return{"messages": [llm_with_tools.invoke([sys_msg]+state["messages"])]}


In [ ]:
from langgraph.graph import START,StateGraph
from langgraph.prebuilt import tools_condition,ToolNode
from IPython.display import display,Image

builder = StateGraph(MessagesState)

builder.add_node("assistant",assistant)
builder.add_node("tools",ToolNode(tools))

builder.add_edge(START,"assistant")
builder.add_conditional_edges("assistant",tools_condition)

builder.add_edge("tools","assistant")
react_graph = builder.complie()

display(Image(react_graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
messages =[HumanMessage(content = "Add 3 and 4.")]
messages = react_graph.invoke({"messages":messages})
for m in messages['messages']:
    m.pretty_print()

In [ ]:
messages = [HumanMessage(content = "Multiply that by 2.")]
messages = react_graph.invoke({"messages":messages})
for m in messages['messages']:
    m.pretty_print()

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()
react_graph_memory = builder.compile(checkpointer = memory)

In [ ]:
config = {"configurable":{"thread_id":"1"}}
messages = [HumanMessage(content = "Add 3 and 4.")]
messages = react_graph_memory.invoke({"messages":messages},config)
for m in messages['messages']:
    m.pretty_print()